# Test 1: Revision momentum in GPT-2 small

**Question:** Does "momentum" build during generation such that a revision cue ("wait, let me reconsider") only sometimes redirects the model — and does that get harder the deeper into the trajectory the cue appears?

| Sub-question | What it asks |
|---|---|
| **SQ1** (binary, per-example) | Does the revision cue cause *real* internal change, or only surface language? |
| **SQ2** (momentum) | Does correction success drop as the revision point moves later in the trajectory? |

**Model:** `gpt2-small` via TransformerLens · CPU-fine · ~500MB download

**Method sketch:**
1. ~10 arithmetic/logic prompts where GPT-2 small errs
2. Append a natural revision phrase, vary *where* in the trajectory it appears
3. `run_with_cache()` → compare answer-token logits before vs after the cue
4. Score: did the predicted top token actually change? Does success decline with depth?

---

## Design discussion (open)

Park open questions here before implementing further.

- What counts as "depth" — token position? number of reasoning steps? length of the wrong answer committed?
- What counts as "real internal change" for SQ1 — top-1 flip only, or also logit mass / rank of the correct answer?
- Prompt construction: forced wrong answer in the prompt vs. letting the model generate the error first?
- Control: same length trajectory *without* a revision cue — so we don't confuse length effects with cue effects?
- GPT-2 small may be too weak at arithmetic for "correction" to be meaningful — is that a feature (clear failures) or a confound?

## Setup

In [ ]:
# pip install transformer_lens torch  # once, if needed

import torch
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()
print(f"device={model.cfg.device}, n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}")

## Sanity check

In [ ]:
out = model.generate("The capital of France is", max_new_tokens=10, temperature=0.0, verbose=False)
print(out)

## Experiment body

Deferred until design questions above are settled. Rough shape:

1. Build prompt bank (wrong arithmetic/logic + revision cue at controlled depth)
2. `logits, cache = model.run_with_cache(tokens)`
3. Compare answer-token distribution pre-cue vs post-cue
4. Aggregate SQ1 flips and SQ2 depth curves